In [ ]:
# Global variables to store model instances
_model = None
_tokenizer = None
_pipeline = None

def get_model():
    global _model, _tokenizer, _pipeline
    
    if _model is not None and _tokenizer is not None and _pipeline is not None:
        return _model, _tokenizer, _pipeline
        
    MODEL_DIR = "/home/guests/andreea_magureanu/projects/rare_disease/models/medgemma-4b-it"
    assert torch.cuda.is_available(), "CUDA GPU required."
    
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    
    # 1) Load tokenizer
    _tokenizer = AutoTokenizer.from_pretrained(
        MODEL_DIR,
        local_files_only=True,
        use_fast=True,
        trust_remote_code=True
    )
    
    # 2) Load model
    _model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        local_files_only=True,
        torch_dtype=dtype,
        trust_remote_code=True
    ).to("cuda")
    
    # 3) Create pipeline
    _pipeline = pipeline(
        "text-generation",
        model=_model,
        tokenizer=_tokenizer,
        device=0,
        torch_dtype=dtype,
        pad_token_id=_tokenizer.eos_token_id,
    )
    
    return _model, _tokenizer, _pipeline

In [9]:
from transformers import pipeline
from PIL import Image
import requests, torch, os

MODEL_DIR = "/home/guests/andreea_magureanu/projects/rare_disease/models/medgemma-4b-it"

pipe = pipeline(
    "image-text-to-text",
    model=MODEL_DIR,
    torch_dtype=torch.bfloat16,   # fine on RTX 3090 (ithor)
    device_map="auto",            # safer than device="cuda"
    local_files_only=True,        # force local load
    trust_remote_code=True        # MedGemma requires this
)

image_url = "https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png"
image = Image.open(requests.get(image_url, headers={"User-Agent": "example"}, stream=True, timeout=20).raw).convert("RGB")

out = pipe(images=image, text="<start_of_image> findings:", max_new_tokens=100)
print(out[0]["generated_text"])


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0
The input data was not formatted as a chat with dicts containing 'role' and 'content' keys, even though this model supports chat. Consider using the chat format for better results. For more information, see https://huggingface.co/docs/transformers/en/chat_templating
Keyword argument `local_files_only` is not a valid argument for this processor and will be ignored.


<start_of_image> findings: heart size is normal. the mediastinal and hilar contours are normal. the pulmonary vasculature is normal. the lungs are clear. there are no pleural effusions or pneumothoraces. there are no acute osseous abnormalities. impression: normal chest radiograph.




















































In [5]:
import pdfplumber
import re 
from typing import List

def parse_pdf_to_sections(pdf_path: str) -> List[str]:
    if pdfplumber is None:
        raise ImportError(
            "pdfplumber is not installed; please install it with pip install pdfplumber"
        )
    sections: List[str] = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            # Replace multiple blank lines with a single delimiter
            cleaned = re.sub(r"\n{2,}", "\n\n", text)
            parts = [p.strip() for p in cleaned.split("\n\n") if p.strip()]
            sections.extend(parts)
    return sections

In [6]:
pdf = "/home/guests/andreea_magureanu/projects/rare_disease_pipeline/paper_filter/JS_paper_new_model.pdf"
sections = parse_pdf_to_sections(pdf)

In [11]:
# print(len(sections))
print(sections[0])

Received:7January2022 Revised:4February2022 Accepted:15February2022
DOI:10.1002/ajmg.c.31963
REVIEW ARTICLE
–
Genotype phenotype correlates in Joubert syndrome: A
review
Simone Gana1 | Valentina Serpieri1 | Enza Maria Valente1,2
1NeurogeneticsResearchCenter,IRCCS
Abstract
MondinoFoundation,Pavia,Italy
2DepartmentofMolecularMedicine, Joubertsyndrome(JS)isageneticallyheterogeneousprimaryciliopathycharacterized
UniversityofPavia,Pavia,Italy by a pathognomonic cerebellar and brainstem malformation, the “molar tooth sign,”
Correspondence and variable organ involvement. Over 40 causative genes have been identified to
Prof.EnzaMariaValente,Departmentof
date, explaining up to 94% of cases. To date, gene-phenotype correlates have been
MolecularMedicine,UniversityofPavia,via
Forlanini14,27100Pavia,Italy, delineated onlyfor a handful of genes, directlytranslating into improved counseling
Email:enzamaria.valente@unipv.it
and clinical care. For instance, JS individuals harboring pathogenic variants

In [12]:
import re
from typing import Dict, List, Tuple, Optional
import pdfplumber

HEADING_RX = re.compile(r"^\s*\d+\s*\|\s+[A-Z][A-Z \-/&()0-9]+$")  # e.g., "1 | INTRODUCTION"

def _page_words(page, x_tol=1.0, y_tol=3.0):
    # Use positioned words (far better than extract_text for spacing)
    return page.extract_words(
        x_tolerance=x_tol,
        y_tolerance=y_tol,
        keep_blank_chars=False,
        use_text_flow=True,
        extra_attrs=["x0","x1","top","bottom"]
    )

def _cluster_columns(words: List[dict], gap=30) -> List[List[dict]]:
    # Simple column clustering: split by large horizontal gaps between clusters of x positions
    if not words:
        return []
    xs = sorted({round(w["x0"],1) for w in words})
    cuts = []
    for i in range(1, len(xs)):
        if xs[i] - xs[i-1] > gap:
            cuts.append((xs[i-1]+xs[i])/2)
    # Assign words to bins
    def col_idx(x):
        i = 0
        for c in cuts:
            if x > c: i += 1
        return i
    cols = {}
    for w in words:
        cols.setdefault(col_idx(w["x0"]), []).append(w)
    # Sort each column top-to-bottom, then left-to-right for ties
    return [sorted(col, key=lambda w: (round(w["top"],1), w["x0"])) for _, col in sorted(cols.items())]

def _lines_from_words(words: List[dict], char_gap=2.5, line_gap=3.0) -> List[str]:
    # Group words on approximately same baseline; insert spaces based on x-gaps
    if not words:
        return []
    lines = []
    current_line: List[dict] = []
    prev = None
    for w in words:
        if prev is None:
            current_line = [w]
        else:
            same_line = abs(w["top"] - prev["top"]) <= line_gap
            if not same_line:
                lines.append(_join_words(current_line, char_gap))
                current_line = [w]
            else:
                current_line.append(w)
        prev = w
    if current_line:
        lines.append(_join_words(current_line, char_gap))
    return lines

def _join_words(words: List[dict], char_gap=2.5) -> str:
    # Join with a space when the horizontal gap between tokens suggests a word break
    words = sorted(words, key=lambda w: w["x0"])
    parts = [words[0]["text"]]
    for a, b in zip(words, words[1:]):
        gap = b["x0"] - a["x1"]
        if gap > char_gap:
            parts.append(" ")
        else:
            # if tokens look glued (e.g., "JS)is"), insert space before capital/number
            if re.match(r"[A-Za-z)]$", parts[-1]) and re.match(r"[A-Z0-9(]", b["text"]):
                parts.append(" ")
        parts.append(b["text"])
    line = "".join(parts)
    # Normalize spaces around punctuation
    line = re.sub(r"\s+([,.;:)\]])", r"\1", line)
    line = re.sub(r"([(\[])\s+", r"\1", line)
    return line.strip()

def _detect_blocks(lines: List[str]) -> Dict[str, List[str]]:
    blocks = {"front_matter": [], "abstract": [], "keywords": [], "body": []}
    mode = "front_matter"
    for ln in lines:
        ln_stripped = ln.strip()
        if re.match(r"^Abstract\b", ln_stripped, flags=re.I):
            mode = "abstract"
            ln_stripped = re.sub(r"^Abstract[:\s-]*", "", ln_stripped, flags=re.I).strip()
            if ln_stripped:
                blocks["abstract"].append(ln_stripped)
            continue
        if re.match(r"^Keywords?\b", ln_stripped, flags=re.I):
            mode = "keywords"
            ln_stripped = re.sub(r"^Keywords?[:\s-]*", "", ln_stripped, flags=re.I).strip()
            if ln_stripped:
                blocks["keywords"].append(ln_stripped)
            continue
        # Wiley body headings e.g. "1 | INTRODUCTION"
        if HEADING_RX.match(ln_stripped):
            mode = "body"
            blocks["body"].append(f"\n## {ln_stripped}")
            continue
        blocks[mode].append(ln_stripped)
    return blocks

def parse_pdf_to_sections(pdf_path: str) -> Dict[str, object]:
    """
    Returns:
      {
        'title': str,
        'authors': str,
        'abstract': str,
        'keywords': List[str],
        'sections': List[str],  # markdown-ish with '## HEADING' markers
        'raw_pages': List[str], # concatenated clean text per page
      }
    """
    pages_clean: List[str] = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            words = _page_words(page)
            columns = _cluster_columns(words)
            col_lines: List[str] = []
            for col in columns:
                col_lines.extend(_lines_from_words(col))
                col_lines.append("")  # paragraph break between columns
            # remove repeating headers/footers (very simple heuristic)
            col_lines = [ln for ln in col_lines if not re.search(r"^©\d{4}|^\s*\d+\s*/\s*\d+\s*$", ln)]
            # collapse blank lines
            text = "\n".join([ln for ln in col_lines if ln is not None]).strip()
            text = re.sub(r"\n{3,}", "\n\n", text)
            pages_clean.append(text)

    # First page: try to guess title + authors (heuristics)
    first = pages_clean[0] if pages_clean else ""
    first_lines = [ln for ln in first.split("\n") if ln.strip()]
    title = ""
    authors = ""
    # Heuristic: find the longest line in first ~15 lines as title
    cand = sorted(first_lines[:15], key=len, reverse=True)
    if cand:
        title = cand[0]
    # Authors often follow the title line; join next 1-3 lines until an affiliation/numbered marker
    try:
        idx = first_lines.index(title)
        auth_lines = []
        for ln in first_lines[idx+1:idx+5]:
            if re.search(r"^\d|\bDepartment\b|\bUniversity\b|Correspondence\b|Abstract\b", ln, re.I):
                break
            auth_lines.append(ln)
        authors = re.sub(r"\s{2,}", " ", " ".join(auth_lines)).strip()
    except ValueError:
        pass

    # Build global lines and split into blocks
    all_lines = []
    for p in pages_clean:
        all_lines.extend(p.split("\n"))
        all_lines.append("")  # page break as blank line
    blocks = _detect_blocks(all_lines)

    # Post-process keywords
    keywords_text = " ".join(blocks["keywords"])
    keywords = [k.strip(" .;:,") for k in re.split(r"[;,•]\s*|\s{2,}", keywords_text) if k.strip()]

    # Body sections: merge consecutive lines; keep headings we injected
    body = "\n".join(blocks["body"])
    # merge lines to paragraphs
    body = re.sub(r"(?<!\n)\n(?![#\n])", " ", body)  # join non-heading line breaks
    body = re.sub(r"\s+\n", "\n", body)

    abstract = " ".join(blocks["abstract"]).strip()
    abstract = re.sub(r"\s+", " ", abstract)

    return {
        "title": title.strip(),
        "authors": authors,
        "abstract": abstract,
        "keywords": keywords,
        "sections": [s for s in body.split("\n\n") if s.strip()],
        "raw_pages": pages_clean,
    }


In [13]:
a = parse_pdf_to_sections(pdf)

In [16]:
print(a["title"])

2Departmentof Molecular Medicine, Joubertsyndrome (J S)isageneticallyheterogeneousprimaryciliopathycharacterized


In [2]:
import requests
from lxml import etree as ET

# Europe PMC gives a clean JATS endpoint by PMCID (often simpler than OAI):
JATS_URL = "https://www.ebi.ac.uk/europepmc/webservices/rest/PMC9314610/fullTextXML"

NS = {
    "j": "http://www.ncbi.nlm.nih.gov/JATS1",
    "xlink": "http://www.w3.org/1999/xlink"
}

def get_jats_xml(url=JATS_URL, timeout=60):
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return ET.fromstring(r.content)

def txt(node):
    return " ".join(" ".join(node.itertext()).split())

DROP_HEADS = {
    "acknowledgements","acknowledgments","funding","funding information",
    "conflict of interest","competing interests","author contributions",
    "data availability","ethics","references","bibliography",
    "supplementary","appendix","correspondence"
}

def parse_title_abs_paras(root):
    # Title
    title_nodes = root.xpath(".//j:article-title", namespaces=NS)
    title = txt(title_nodes[0]) if title_nodes else ""

    # Abstract (all abstract parts joined)
    abs_nodes = root.xpath(".//j:abstract", namespaces=NS)
    abstract = txt(abs_nodes[0]) if abs_nodes else ""

    # Body paragraphs: skip unwanted sections
    paras = []
    for sec in root.xpath(".//j:body//j:sec", namespaces=NS):
        head = " ".join(sec.xpath("./j:title//text()", namespaces=NS)).lower().strip()
        if any(bad in head for bad in DROP_HEADS):
            continue
        for p in sec.xpath(".//j:p", namespaces=NS):
            t = txt(p)
            if len(t.split()) >= 5:
                paras.append(t)
    # Fallback: any body <p> if sec-based missed
    if not paras:
        for p in root.xpath(".//j:body//j:p", namespaces=NS):
            t = txt(p)
            if len(t.split()) >= 5:
                paras.append(t)
    return title, abstract, paras

if __name__ == "__main__":
    root = get_jats_xml()
    title, abstract, paras = parse_title_abs_paras(root)
    print("\nTITLE:\n", title, "\n")
    print("ABSTRACT:\n", abstract, "\n")
    print("FIRST 3 PARAGRAPHS:\n")
    for i, p in enumerate(paras[:3], 1):
        print(f"[{i}] {p}\n")
    print(f"Total paragraphs found: {len(paras)}")



TITLE:
  

ABSTRACT:
  

FIRST 3 PARAGRAPHS:

Total paragraphs found: 0


In [1]:
import requests
from lxml import etree as ET

PMCID = "PMC9314610"

URLS = [
    f"https://www.ebi.ac.uk/europepmc/webservices/rest/{PMCID}/fullTextXML",
    f"https://www.ncbi.nlm.nih.gov/pmc/oai/oai.cgi?verb=GetRecord&identifier=oai:pubmedcentral.nih.gov:{PMCID}&metadataPrefix=pmc",
]

DROP_HEADS = {
    "acknowledgements","acknowledgments","funding","funding information",
    "conflict of interest","competing interests","author contributions",
    "data availability","ethics","references","bibliography",
    "supplementary","appendix","correspondence"
}

def fetch_xml():
    last_err = None
    for url in URLS:
        try:
            r = requests.get(url, timeout=90)
            r.raise_for_status()
            # Quick sanity check: must contain an <article> element
            if b"<article" in r.content:
                return ET.fromstring(r.content)
            # Some wrappers (OAI) put <article> deeper; still okay
            try:
                root = ET.fromstring(r.content)
                if root.xpath("//*[local-name()='article']"):
                    return root
            except ET.XMLSyntaxError as e:
                last_err = e
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Failed to fetch XML for {PMCID}: {last_err}")

def norm_text(node):
    return " ".join(" ".join(node.itertext()).split())

def parse_title_abs_paras(root):
    # Find the article node regardless of namespaces/wrappers
    article_nodes = root.xpath("//*[local-name()='article']")
    if not article_nodes:
        return "", "", []
    article = article_nodes[0]

    # Title
    title_nodes = article.xpath(".//*[local-name()='article-title']")
    title = norm_text(title_nodes[0]) if title_nodes else ""

    # Abstract (join all parts)
    abs_nodes = article.xpath(".//*[local-name()='abstract']")
    abstract = norm_text(abs_nodes[0]) if abs_nodes else ""

    # Body → sections → paragraphs (skip unwanted heads)
    paras = []
    for sec in article.xpath(".//*[local-name()='body']//*[local-name()='sec']"):
        head = " ".join(sec.xpath(".//*[local-name()='title']/text()")).strip().lower()
        if any(bad in head for bad in DROP_HEADS):
            continue
        for p in sec.xpath(".//*[local-name()='p']"):
            t = norm_text(p)
            if len(t.split()) >= 5:
                paras.append(t)

    # Fallback: any <p> under body if secs were missed
    if not paras:
        for p in article.xpath(".//*[local-name()='body']//*[local-name()='p']"):
            t = norm_text(p)
            if len(t.split()) >= 5:
                paras.append(t)

    return title, abstract, paras

if __name__ == "__main__":
    root = fetch_xml()
    title, abstract, paras = parse_title_abs_paras(root)

    print("\nTITLE:\n", title, "\n")
    print("ABSTRACT (first 500 chars):\n", abstract[:500], "...\n")
    print("FIRST 3 PARAGRAPHS:\n")
    for i, p in enumerate(paras[:3], 1):
        print(f"[{i}] {p}\n")
    print(f"Total paragraphs found: {len(paras)}")



TITLE:
 Genotype–phenotype correlates in Joubert syndrome: A review 

ABSTRACT (first 500 chars):
 Abstract Joubert syndrome (JS) is a genetically heterogeneous primary ciliopathy characterized by a pathognomonic cerebellar and brainstem malformation, the “molar tooth sign,” and variable organ involvement. Over 40 causative genes have been identified to date, explaining up to 94% of cases. To date, gene‐phenotype correlates have been delineated only for a handful of genes, directly translating into improved counseling and clinical care. For instance, JS individuals harboring pathogenic varian ...

FIRST 3 PARAGRAPHS:

[1] Joubert syndrome (JS) is a rare congenital neurodevelopmental primary ciliopathy with a population‐based prevalence reaching 1.7 per 100,000 in the age range 0–19 years (Nuovo et al., 2020 ). First described by Dr Marie Joubert about 50 years ago (Joubert, Eisenring, Robb, & Andermann, 1969 ), JS is now diagnosed upon recognition of a pathognomonic malformation of th

In [4]:
# -*- coding: utf-8 -*-
import os, re, json, math
import torch
from typing import Dict, Any, List
from transformers import AutoTokenizer, AutoModelForCausalLM

# -----------------------------
# 0) Config
# -----------------------------
MODEL_DIR = "/home/guests/andreea_magureanu/projects/rare_disease/models/medgemma-4b-it"
assert torch.cuda.is_available(), "CUDA GPU required."

# Pick the safest dtype for your GPU
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
DEVICE = "cuda"

# Generation defaults (tune as needed)
GEN_KW = dict(
    max_new_tokens=512,          # allow enough room for entities + relations
    do_sample=True,              # sampling generally helps IE coverage
    temperature=0.2,             # low but non-zero = less hallucination than 0.7
    top_p=0.9,
    repetition_penalty=1.05,     # reduce key repetition without crushing recall
)

# -----------------------------
# 1) Load tokenizer & model
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    use_fast=True,
    trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    torch_dtype=DTYPE,
    trust_remote_code=True
).to(DEVICE)

# Ensure pad/eos are set safely
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
PAD = tokenizer.pad_token_id
EOS = tokenizer.eos_token_id

# -----------------------------
# 2) Prompt utilities
# -----------------------------
SYSTEM = (
  "You are a strict medical information extraction agent. "
  "Output ONLY a single valid JSON object with keys 'entities' and 'relations'. "
  "Each entity is an object with keys: 'name' (string), 'type' "
  "(one of 'disease','gene','genotype','phenotype','treatment'), and optional 'aliases' (array of strings). "
  "Each relation is an object with keys: 'subject','predicate','object' (all strings referencing entity names). "
  "Return [] for any empty array. No commentary, no markdown—JSON only."
)

# A tiny few-shot showing aliases + relations (very helpful!)
FEWSHOT_USER = (
  "Title: Duchenne muscular dystrophy case report\n"
  "Abstract: Deletions in the DMD gene cause Duchenne muscular dystrophy (DMD). "
  "Patients show progressive muscle weakness and elevated CK. Corticosteroids improve motor outcomes."
)
FEWSHOT_ASSISTANT = {
  "entities": [
    {"name": "Duchenne muscular dystrophy", "type": "disease", "aliases": ["DMD"]},
    {"name": "DMD", "type": "gene", "aliases": ["dystrophin"]},
    {"name": "progressive muscle weakness", "type": "phenotype", "aliases": []},
    {"name": "elevated creatine kinase", "type": "phenotype", "aliases": ["elevated CK"]},
    {"name": "corticosteroids", "type": "treatment", "aliases": ["steroids"]}
  ],
  "relations": [
    {"subject": "DMD", "predicate": "causes", "object": "Duchenne muscular dystrophy"},
    {"subject": "Duchenne muscular dystrophy", "predicate": "associated_with", "object": "progressive muscle weakness"},
    {"subject": "Duchenne muscular dystrophy", "predicate": "associated_with", "object": "elevated creatine kinase"},
    {"subject": "corticosteroids", "predicate": "improves", "object": "motor outcomes"}
  ]
}

def build_messages(title: str, abstract: str) -> List[Dict[str, str]]:
    user_now = (
        f"Title: {title}\nAbstract: {abstract}\n\n"
        "Identify ONLY these entities: disease, gene, genotype, phenotype, treatment.\n"
        "Include 'aliases' when present (synonyms, abbreviations, HGNC symbols, common short forms found in text).\n"
        "Return exactly one JSON with 'entities' and 'relations'. If none, return empty arrays.\n"
    )
    return [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": FEWSHOT_USER},
        {"role": "assistant", "content": json.dumps(FEWSHOT_ASSISTANT, separators=(",", ":"))},
        {"role": "user", "content": user_now},
    ]

# -----------------------------
# 3) Robust generator
# -----------------------------
def run_ie(title: str, abstract: str, max_retries: int = 2) -> Dict[str, Any]:
    """
    Generates JSON using chat template correctly and parses it robustly.
    """
    messages = build_messages(title, abstract)

    # IMPORTANT: For MedGemma + HF, use apply_chat_template -> string -> then tokenize yourself.
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True  # adds the assistant "starts here" marker
    )
    # Tokenize to tensors (THIS avoids your earlier `** must be a mapping` error)
    enc = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
        padding=False
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    for attempt in range(max_retries + 1):
        with torch.no_grad():
            out_ids = model.generate(
                **enc,
                eos_token_id=EOS,
                pad_token_id=PAD,
                **GEN_KW
            )
        # Slice off the prompt
        new_tokens = out_ids[0, enc["input_ids"].shape[-1]:]
        out_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        data = try_parse_json(out_text)
        if data is not None:
            data = postprocess_schema(data, abstract)
            return data

        # Last resort: try to salvage the last {...} block
        m = re.search(r"\{[\s\S]*\}\s*$", out_text)
        if m:
            data = try_parse_json(m.group(0))
            if data is not None:
                data = postprocess_schema(data, abstract)
                return data

        # If we got here, retry once with slightly higher max_new_tokens
        if attempt < max_retries:
            GEN_KW["max_new_tokens"] = int(GEN_KW["max_new_tokens"] * 1.25)

    # If completely failed, return empty schema (never crash the pipeline)
    return {"entities": [], "relations": []}

# -----------------------------
# 4) Parsing + Post-processing
# -----------------------------
def try_parse_json(text: str):
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def dedup_entities(entities: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = {}
    result = []
    for e in entities or []:
        name = (e.get("name") or "").strip()
        etype = (e.get("type") or "").strip().lower()
        if not name or etype not in {"disease","gene","genotype","phenotype","treatment"}:
            continue
        key = (name.lower(), etype)
        if key not in seen:
            # normalize aliases
            aliases = e.get("aliases") or []
            aliases = [a.strip() for a in aliases if isinstance(a, str) and a.strip()]
            e["aliases"] = sorted(set(aliases))
            result.append(e)
            seen[key] = True
    return result

def postprocess_schema(data: Dict[str, Any], raw_text: str) -> Dict[str, Any]:
    # Ensure keys exist
    entities = data.get("entities") or []
    relations = data.get("relations") or []

    # Deduplicate + normalize entities
    entities = dedup_entities(entities)

    # Light alias heuristic: harvest acronyms in parentheses that follow a long name
    # e.g., "Joubert syndrome (JS)" -> add "JS" as alias to "Joubert syndrome"
    for i, e in enumerate(entities):
        name = e["name"]
        paren_acros = re.findall(rf"{re.escape(name)}\s*\(([^)]+)\)", raw_text)
        for ac in paren_acros:
            ac = ac.strip()
            if 2 <= len(ac) <= 15 and " " not in ac:
                e["aliases"] = sorted(set([*e.get("aliases", []), ac]))

    # Relations: keep only if strings and non-empty
    clean_rel = []
    for r in relations:
        s = (r.get("subject") or "").strip()
        p = (r.get("predicate") or "").strip()
        o = (r.get("object") or "").strip()
        if s and p and o:
            clean_rel.append({"subject": s, "predicate": p, "object": o})

    return {"entities": entities, "relations": clean_rel}

# -----------------------------
# 5) Example usage
# -----------------------------
if __name__ == "__main__":
    result = run_ie(title, abstract)
    print(json.dumps(result, indent=2, ensure_ascii=False))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

{
  "entities": [],
  "relations": []
}


In [1]:
import torch
print("CUDA available? ", torch.cuda.is_available())
print("GPU count: ", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name: ", torch.cuda.get_device_name(0))


CUDA available?  True
GPU count:  2
GPU name:  NVIDIA TITAN V


In [1]:
import os, torch
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("SLURM_JOB_GPUS      =", os.environ.get("SLURM_JOB_GPUS"))
print("SLURM_STEP_GPUS     =", os.environ.get("SLURM_STEP_GPUS"))
print("torch sees", torch.cuda.device_count(), "GPU(s)")
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


CUDA_VISIBLE_DEVICES = 0
SLURM_JOB_GPUS      = 0
SLURM_STEP_GPUS     = None
torch sees 1 GPU(s)
0 NVIDIA TITAN V


In [ ]:

# 1) Imports
import torch, re, json
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# 2) Config
MODEL_DIR = "/home/guests/andreea_magureanu/projects/rare_disease/models/medgemma-4b-it"
assert torch.cuda.is_available(), "CUDA GPU required."

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# 3) Load tokenizer & model locally (no internet/cache)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    use_fast=True,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    device_map=None,
    torch_dtype=dtype,
    trust_remote_code=True
)

# 4) Build text-generation pipeline (do NOT pass local_files_only here)
gen = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map=None,
    torch_dtype=dtype,
    pad_token_id=tokenizer.eos_token_id,  # safer than 0
)

print("Loaded:", dtype)



# 7. Combine title and abstract into a single text block
text = f"Title: {title}\nAbstract: {abstract}"

# 8. Craft the prompt (as given in your question)
prompt = (
    "You are a medical information extraction agent. "
    "Given the following text, identify all diseases, causative genes, "
    "phenotypes, genotypes and treatments mentioned. "
    "Return your answer as a JSON object with two keys: 'entities' and 'relations'. "
    "The 'entities' value should be a list of objects with 'name' and 'type' keys, "
    "where type is one of 'disease', 'gene', 'phenotype', 'genotype', 'treatment'. "
    "The 'relations' value should be a list of objects with 'subject', 'predicate' and 'object' keys, "
    "describing relations between the entities (e.g. 'causes', 'associated_with', etc.). "
    "If no relations are present, return an empty list for 'relations'. "
    "Provide only the JSON object as output without explanation.\n\n"
    f"Text: {text}\n"
)

# 9. Run the model on the prompt
out = gen(prompt, max_new_tokens=160, do_sample=False)[0]["generated_text"]
print("Raw output:\n", out)

# 10. (Optional) Extract the JSON object from the model output
import re, json

match = re.search(r"\{[\s\S]*\}\s*$", out)
if match:
    try:
        extracted = json.loads(match.group(0))
        print("\nExtracted JSON:\n", json.dumps(extracted, indent=2))
    except json.JSONDecodeError:
        print("Model returned non-valid JSON; additional cleanup may be required.")
else:
    print("No JSON object detected in the output.")


`torch_dtype` is deprecated! Use `dtype` instead!
/home/guests/andreea_magureanu/.conda/envs/rare_dis/lib/python3.11/site-packages/accelerate/utils/modeling.py:1582: UserWarning: Current model requires 33282 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cpu


Loaded: torch.bfloat16
Raw output:
 You are a medical information extraction agent. Given the following text, identify all diseases, causative genes, phenotypes, genotypes and treatments mentioned. Return your answer as a JSON object with two keys: 'entities' and 'relations'. The 'entities' value should be a list of objects with 'name' and 'type' keys, where type is one of 'disease', 'gene', 'phenotype', 'genotype', 'treatment'. The 'relations' value should be a list of objects with 'subject', 'predicate' and 'object' keys, describing relations between the entities (e.g. 'causes', 'associated_with', etc.). If no relations are present, return an empty list for 'relations'. Provide only the JSON object as output without explanation.

Text: Title: Genotype–phenotype correlates in Joubert syndrome: A review
Abstract: Abstract Joubert syndrome (JS) is a genetically heterogeneous primary ciliopathy characterized by a pathognomonic cerebellar and brainstem malformation, the “molar tooth sign,

In [2]:
import torch, json, re
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_DIR = "/home/guests/andreea_magureanu/projects/rare_disease/models/medgemma-4b-it"
assert torch.cuda.is_available(), "CUDA GPU required."

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

tok = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True, use_fast=True, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, local_files_only=True, torch_dtype=dtype, trust_remote_code=True).to("cuda")

gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tok,
    device=0,
    torch_dtype=dtype,
    pad_token_id=tok.eos_token_id,
)

# ---- Your paper ----
title = "Genotype–phenotype correlates in Joubert syndrome: A review"
abstract = ("Joubert syndrome (JS) is a primary ciliopathy... TMEM67 -> higher risk of liver fibrosis; "
            "NPHP1/RPGRIP1L/TMEM237 -> renal involvement; CEP290/AHI1 -> retinal dystrophy; CEP290 -> CKD.")

# ---- Few-shot with explicit schema & aliases ----
system = (
  "You are a strict medical IE agent. Output ONLY valid minified JSON with keys "
  "'entities' and 'relations'. Entities must include optional 'aliases' (array). No prose."
)

fewshot_input = (
  "Title: Duchenne muscular dystrophy case report\n"
  "Abstract: Deletions in the DMD gene cause Duchenne muscular dystrophy (DMD). "
  "Patients show progressive muscle weakness and elevated CK. Corticosteroids improve motor outcomes."
)
fewshot_output = {
  "entities": [
    {"name": "Duchenne muscular dystrophy", "type": "disease", "aliases": ["DMD"]},
    {"name": "DMD", "type": "gene", "aliases": ["dystrophin"]},
    {"name": "progressive muscle weakness", "type": "phenotype", "aliases": []},
    {"name": "elevated creatine kinase", "type": "phenotype", "aliases": ["elevated CK"]},
    {"name": "corticosteroids", "type": "treatment", "aliases": ["steroids"]}
  ],
  "relations": [
    {"subject": "DMD", "predicate": "causes", "object": "Duchenne muscular dystrophy"},
    {"subject": "Duchenne muscular dystrophy", "predicate": "associated_with", "object": "progressive muscle weakness"},
    {"subject": "Duchenne muscular dystrophy", "predicate": "associated_with", "object": "elevated creatine kinase"},
    {"subject": "corticosteroids", "predicate": "improves", "object": "motor outcomes"}
  ]
}

user_text = (
  f"Title: {title}\nAbstract: {abstract}\n\n"
  "Identify all diseases, genes, phenotypes, genotypes, treatments in this text. "
  "Return a single JSON with 'entities' (each may have 'aliases') and 'relations'. "
  "If none, return empty arrays. JSON only, one object, no explanation."
)

messages = [
  {"role": "system", "content": system},
  {"role": "user", "content": fewshot_input},
  {"role": "assistant", "content": json.dumps(fewshot_output, separators=(",",":"))},
  {"role": "user", "content": user_text}
]

prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

out = gen(
    prompt,
    max_new_tokens=320,        # bump length so it reaches relations
    do_sample=False,           # deterministic; you can try temperature=0.2 later
    return_full_text=False,    # don't echo the prompt
)[0]["generated_text"]

# Try to parse JSON (expecting a clean single-object response)
try:
    data = json.loads(out)
except json.JSONDecodeError:
    m = re.search(r"\{[\s\S]*\}\s*$", out)
    data = json.loads(m.group(0)) if m else {"entities": [], "relations": []}

print(json.dumps(data, indent=2))


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{
  "entities": [],
  "relations": []
}


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_DIR = "/home/guests/andreea_magureanu/projects/rare_disease/models/medgemma-4b-it"
assert torch.cuda.is_available(), "CUDA GPU required."

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# 1) Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    use_fast=True,
    trust_remote_code=True
)

# 2) Load model (no device_map), then move to GPU explicitly
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    torch_dtype=dtype,
    trust_remote_code=True
)

model = model.to("cuda")   # <-- force entire model onto GPU

# 3) Build pipeline (point explicitly to GPU device 0)
gen = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0,                        # 0 = first CUDA GPU
    torch_dtype=dtype,
    pad_token_id=tokenizer.eos_token_id,
)

# 4) Print confirmation
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Model loaded on device: {next(model.parameters()).device}")
print(f"Using dtype: {dtype}")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


CUDA available: True
Model loaded on device: cuda:0
Using dtype: torch.bfloat16


In [4]:
# 1) Drop references
try:
    del gen
except: pass
try:
    del model
except: pass
try:
    del tokenizer
except: pass

# 2) Garbage-collect Python objects
import gc, torch, os
gc.collect()

# 3) Free CUDA caches
torch.cuda.empty_cache()

# 4) (Optional) reduce fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# or:
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_DIR = "/home/guests/andreea_magureanu/projects/rare_disease/models/medgemma-4b-it"
assert torch.cuda.is_available(), "CUDA GPU required."

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

messages = [
    {"role": "system", "content": "You are a medical IE agent. Reply with JSON only."},
    {"role": "user", "content": f"Title: {title}\nAbstract: {abstract}\nExtract entities and relations."}
]


# 1) Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    use_fast=True,
    trust_remote_code=True
)

prompt_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,   # adds the “assistant starts here” marker
    tokenize=True,
    return_tensors="pt"
)

# 2) Load model (no device_map), then move to GPU explicitly
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    torch_dtype=dtype,
    trust_remote_code=True
)

model = model.to("cuda")   # <-- force entire model onto GPU

output_ids = model.generate(
    **prompt_ids,
    max_new_tokens=220,      # up to ~220 tokens AFTER your abstract
    do_sample=True,          # enable sampling
    temperature=0.3,         # modest randomness
    top_p=0.9                # nucleus sampling
)
input_len = prompt_ids["input_ids"].shape[-1]
new_ids = output_ids[0][input_len:]
text_output = tokenizer.decode(new_ids, skip_special_tokens=True)



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

TypeError: transformers.generation.utils.GenerationMixin.generate() argument after ** must be a mapping, not Tensor

In [3]:
# 1) Drop references
try:
    del gen
except: pass
try:
    del model
except: pass
try:
    del tokenizer
except: pass

# 2) Garbage-collect Python objects
import gc, torch, os
gc.collect()

# 3) Free CUDA caches
torch.cuda.empty_cache()

# 4) (Optional) reduce fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# or:
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

In [ ]:
# Get or initialize the model
model, tokenizer, gen = get_model()

# Use it for inference
messages = [
    {"role": "system", "content": "You are a medical IE agent. Reply with JSON only."},
    {"role": "user", "content": f"Title: {title}\nAbstract: {abstract}\nExtract entities and relations."}
]

prompt_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt"
).to("cuda")

output_ids = model.generate(
    prompt_ids["input_ids"],
    max_new_tokens=220,
    do_sample=True,
    temperature=0.3,
    top_p=0.9
)